# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maheen-armghan/flyrank-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Classification**, applied downstream as a ranking.

My lane is Refresh / Content Opportunity Scoring. The core prediction is binary — is a page declining or not — which makes classification the right label for the model type. The classifier's output probability is then used to rank pages, producing the actual deliverable (a prioritized review queue), but the model itself is trained and evaluated as a classifier, not a ranker directly.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

REPO_URL = "https://github.com/maheen-armghan/flyrank-internship.git"
REPO_NAME = "flyrank-internship"

if not os.path.exists(REPO_NAME):
    !git clone {REPO_URL}

os.chdir(REPO_NAME)
print("Now in:", os.getcwd())

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (328/328), done.
remote: Compressing objects: 100% (144/144), done.
remote: Total 328 (delta 176), reused 293 (delta 156), pack-reused 0 (from 0)
Receiving objects: 100% (328/328), 1.91 MiB | 13.32 MiB/s, done.
Resolving deltas: 100% (176/176), done.
Now in: /content/flyrank-internship


In [2]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df["trend_direction"].value_counts())

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target (proxy): `trend_direction == "down"`.**

This is a proxy, not the ideal target — it's a bucket calculated from the current window, not a true future outcome. A stronger version (noted in the lane guide) would be: given a page's signals over the prior 90 days, does it decline over the next 30 days? I'm using the current-window proxy for this assignment because it's what the starter dataset ships with, but I'm naming this limitation explicitly rather than treating the proxy as the real target.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(df["is_declining_label"].value_counts(normalize=True))

is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Metric: Precision@K** (specifically Precision@50), matching a reviewer's real capacity.

Since the output is a ranked review queue and reviewers can only check a limited number of pages, Precision@50 answers the practically relevant question: "of the top 50 pages the model flags, how many are actually declining?" This matches how the output will actually be used far better than a generic metric like accuracy would, which doesn't account for the fact that only the top of the ranking matters operationally.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# demonstrate with a placeholder score (e.g. impressions_90d) just to show the metric works
demo_score = df["impressions_90d"].values
demo_labels = df["is_declining_label"].values
print("Demo Precision@50 (using impressions_90d as a naive score):",
      round(precision_at_k(demo_score, demo_labels, 50), 3))

Demo Precision@50 (using impressions_90d as a naive score): 0.42


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one content page** (`content_id`), after deduplication. Each row represents a single page's aggregated 90-day performance snapshot — not a page-day, not a client, not a query.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df_filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df_filtered = df_filtered.drop_duplicates(subset="content_id")

print("Rows (= unique pages):", len(df_filtered))
df_filtered[["content_id", "impressions_90d", "avg_position", "ctr",
             "content_age_days", "trend_direction", "is_declining_label"]].head(5)

Rows (= unique pages): 30000


,content_id,impressions_90d,avg_position,ctr,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,3803,10.6,0.76,187,down,1
1,content_a1fb4e703a9e,15320,20.3,0.05,445,down,1
2,content_9aa793d4d895,12581,36.5,0.09,141,down,1
3,content_331d6c4de07b,11751,6.2,0.49,463,stable,0
4,content_d99b7a2d90ca,19140,44.0,0.13,263,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In Week 1's exploration, `days_since_last_update` had an identical median (20.0 days) for both declining and non-declining pages — a simple hand rule like "flag stale pages" would fail to separate the two groups on this signal alone. Notebook 02's tree-vs-hand-rule comparison also showed a learned model outperforming a single heuristic at Precision@50. Decline is likely driven by a combination of signals (position, CTR, impressions, age) interacting together — exactly the kind of multi-signal pattern a model can weigh automatically, but a human-written if/else rule struggles to capture without becoming unreadably complex.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.